# Outlier Context Inspector (v5.3)

**작성일**: 2026-01-11
**목적**: 탐지된 이상치(Outlier) 이벤트의 **직전/직후** 상황을 함께 조회하여 원인을 파악합니다.

### 주요 기능
- `preprocessed_data.csv` (전체 데이터)와 `preprocessed_data_outliers.csv` (이상치 목록)를 로드합니다.
- 이상치 이벤트의 `game_id`와 `action_id`를 기준으로 **앞뒤 1개(총 3개)** 이벤트를 추출하여 보여줍니다.
- 이를 통해 단순 측정 오류인지, 데이터 누락(Gap)인지, 혹은 복원 가능한 패턴인지 판단할 수 있습니다.

In [1]:
import pandas as pd
import numpy as np

# 1. 데이터 로드
data_path = '../data/refined/v5.3/preprocessed_data.csv'
outlier_path = '../data/refined/v5.3/preprocessed_data_outliers.csv'

print("Loading full data...")
df = pd.read_csv(data_path, encoding='utf-8-sig')

print("Loading outlier list...")
outliers = pd.read_csv(outlier_path, encoding='utf-8-sig')

print(f"Full Data: {len(df):,} rows")
print(f"Outliers: {len(outliers):,} rows")

# 정렬 보장 (Context 추출을 위해 필수)
df = df.sort_values(['game_id', 'period_id', 'time_seconds', 'action_id']).reset_index(drop=True)

# 빠른 검색을 위한 인덱스 매핑 생성 (game_id, action_id) -> dataframe index
index_map = dict(zip(zip(df['game_id'], df['action_id']), df.index))
print("Index map created.")

Loading full data...
Loading outlier list...
Full Data: 465,787 rows
Outliers: 3,126 rows
Index map created.


## 2. 이상치 문맥 추출 함수 정의
이상치 이벤트 전후 1개를 포함한 Context DataFrame을 생성합니다.

In [2]:
def get_outlier_context(outlier_row, full_df, idx_map, context_window=1):
    game_id = outlier_row['game_id']
    action_id = outlier_row['action_id']
    
    # 전체 데이터에서의 인덱스 찾기
    target_idx = idx_map.get((game_id, action_id))
    
    if target_idx is None:
        return None
    
    # 앞뒤 범위 설정 (같은 게임 내에서만)
    start_idx = max(0, target_idx - context_window)
    end_idx = min(len(full_df) - 1, target_idx + context_window)
    
    # Context 데이터 추출
    context_df = full_df.iloc[start_idx : end_idx + 1].copy()
    
    # 다른 게임 데이터가 섞이지 않도록 필터링
    context_df = context_df[context_df['game_id'] == game_id]
    
    # 시각화를 위한 플래그 추가 (Is Target?)
    context_df['is_target'] = context_df['action_id'] == action_id
    
    # 이상치 정보 병합 (속도 등)
    # 원본 df에는 없을 수 있는 계산된 속도(internal_speed, gap_speed)를 보기 위해
    # outlier_row의 정보를 target 행에만 overwrite 하거나 별도 표시
    if 'internal_speed' not in context_df.columns:
        context_df['internal_speed'] = np.nan
    if 'gap_speed' not in context_df.columns:
        context_df['gap_speed'] = np.nan
        
    # Target 행에 계산된 값 넣기
    context_df.loc[context_df['is_target'], 'internal_speed'] = outlier_row['internal_speed']
    context_df.loc[context_df['is_target'], 'gap_speed'] = outlier_row['gap_speed']
    context_df.loc[context_df['is_target'], 'outlier_type'] = outlier_row['outlier_type']
    
    return context_df

## 3. 샘플 조회 (Top 20 Extreme Internal Speed)
**Internal Speed(단일 이벤트 과속)** 가 가장 높은 상위 20개 케이스를 주변 문맥과 함께 살펴봅니다.
주로 지속 시간(`dt`)이 0에 가깝거나, 좌표가 튀는 경우입니다.

In [5]:
target_outliers = outliers.sort_values('internal_speed', ascending=False).head(20)

cols_to_show = ['game_id', 'time_seconds', 'player_name_ko', 'type_name', 'start_x', 'start_y', 'end_x', 'end_y', 'internal_speed', 'outlier_type']

for i, (_, row) in enumerate(target_outliers.iterrows()):
    ctx = get_outlier_context(row, df, index_map, context_window=1)
    if ctx is not None:
        print(f"\n[Case {i+1}] Game: {row['game_id']} | Type: {row['outlier_type']} | Speed: {row['internal_speed']:.2f} m/s")
        
        # 스타일링 출력 (Target 행 강조)
        display(ctx[cols_to_show].style.apply(lambda x: ['background: #ffcccc; color : black' if v else '' for v in ctx['is_target']], axis=0))
    else:
        print(f"Could not find context for Action ID {row['action_id']}")


[Case 1] Game: 126416 | Type: Internal | Speed: 33705.92 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,internal_speed,outlier_type
319602,126416,664.333000,박정인,Foul,0.255158,0.939662,0.255158,0.939662,nan,nan
319603,126416,711.829000,김강국,Pass_Freekick,0.241552,0.956991,0.152618,0.480718,33705.920535,Internal
319604,126416,711.830000,김영빈,Foul,0.152618,0.480718,0.152618,0.480718,nan,nan



[Case 2] Game: 126336 | Type: Internal | Speed: 5271.64 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,internal_speed,outlier_type
128112,126336,2301.382000,조현우,Pass,0.028466,0.498117,0.184145,0.924470,nan,nan
128113,126336,2304.394000,심상민,Clearance,0.184145,0.924470,0.478818,0.827898,5271.644814,Internal
128114,126336,2304.400000,유리 조나탄,Tackle,0.179981,0.897466,0.179981,0.897466,nan,nan



[Case 3] Game: 126469 | Type: Internal | Speed: 5169.74 m/s


...

,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,internal_speed,outlier_type
298108,126407,1466.813000,이정택,Carry,0.439706,0.954182,0.423447,0.940168,nan,nan
298109,126407,1467.593000,이정택,Pass,0.423447,0.940168,0.350731,0.849809,1400.071226,Internal
298110,126407,1467.600000,전병관,Tackle,0.426586,0.936016,0.426586,0.936016,nan,nan



[Case 19] Game: 126344 | Type: Internal | Speed: 1347.44 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,internal_speed,outlier_type
147703,126344,2863.200000,티아고,Interception,0.681117,0.810253,0.681117,0.810253,nan,nan
147704,126344,2863.201000,티아고,Carry,0.681117,0.810253,0.670851,0.692423,1347.437154,Internal
147705,126344,2863.207000,한찬희,Carry,0.678873,0.812703,0.670851,0.692423,nan,nan



[Case 20] Game: 126390 | Type: Internal | Speed: 1339.38 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,internal_speed,outlier_type
258500,126390,508.000000,헤이스,Take-On,0.482097,0.137824,0.482097,0.137824,nan,nan
258501,126390,508.001000,헤이스,Carry,0.482097,0.137824,0.877121,0.296634,1339.384185,Internal
258502,126390,508.033000,임종은,Tackle,0.486718,0.153609,0.486718,0.153609,nan,nan


## 4. 샘플 조회 (Top 20 Extreme Gap Speed)
**Gap Speed(이벤트 간 순간이동)** 가 가장 높은 상위 20개 케이스를 살펴봅니다.
이전 이벤트 종료 위치와 현재 이벤트 시작 위치가 극단적으로 다른 경우(데이터 누락 가능성)입니다.

In [ ]:
target_outliers = outliers.sort_values('gap_speed', ascending=False).head(20)

cols_to_show = ['game_id', 'time_seconds', 'player_name_ko', 'type_name', 'start_x', 'start_y', 'end_x', 'end_y', 'gap_speed', 'outlier_type']

for i, (_, row) in enumerate(target_outliers.iterrows()):
    ctx = get_outlier_context(row, df, index_map, context_window=1)
    if ctx is not None:
        print(f"\n[Case {i+1}] Game: {row['game_id']} | Type: {row['outlier_type']} | Gap Speed: {row['gap_speed']:.2f} m/s")
        display(ctx[cols_to_show].style.apply(lambda x: ['background: #ccffff; color: black' if v else '' for v in ctx['is_target']], axis=0))


[Case 1] Game: 126302 | Type: Gap | Gap Speed: 9572.35 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,gap_speed,outlier_type
47907,126302,2164.266000,아론,Offside,0.034575,0.422063,0.034575,0.422063,nan,nan
47908,126302,2164.267000,안톤,Duel,0.062450,0.556091,0.034575,0.422063,9572.347858,Gap
47909,126302,2164.267000,안톤,Pass,0.062450,0.556091,0.034575,0.422063,nan,nan



[Case 2] Game: 126469 | Type: Gap | Gap Speed: 5641.33 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,gap_speed,outlier_type
439066,126469,944.766000,무고사,Pass,0.603337,0.360029,0.631104,0.297247,nan,nan
439067,126469,944.767000,전민광,Tackle,0.600338,0.365259,0.600338,0.365259,5641.329720,Gap
439068,126469,946.166000,신진호,Pass,0.631104,0.297247,0.529550,0.726351,nan,nan



[Case 3] Game: 126336 | Type: Gap | Gap Speed: 5288.75 m/s


...


[Case 19] Game: 126342 | Type: Gap | Gap Speed: 1618.58 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,gap_speed,outlier_type
142584,126342,1428.467000,어정원,Tackle,0.728282,0.727353,0.728282,0.727353,nan,nan
142585,126342,1428.468000,이상헌,Carry,0.725431,0.703961,0.689326,0.713256,1618.579602,Gap
142586,126342,1429.433000,이상헌,Pass,0.689326,0.713256,0.583870,0.862667,nan,nan



[Case 20] Game: 126347 | Type: Gap | Gap Speed: 1596.01 m/s


,game_id,time_seconds,player_name_ko,type_name,start_x,start_y,end_x,end_y,gap_speed,outlier_type
153518,126347,142.620000,박세진,Pass,0.392484,0.356381,0.588137,0.301807,nan,nan
153519,126347,142.633000,서진수,Tackle,0.394082,0.359357,0.394082,0.359357,1596.013722,Gap
153520,126347,145.790000,고명석,Pass,0.588137,0.301807,0.886764,0.483519,nan,nan


: 

## 5. 분석 제언
- **Pass Received 이슈**: 수신 이벤트 직전/직후에 좌표가 튀는 현상이 잦은지 확인 필요.
- **동일 시간(dt=0) 이슈**: `time_seconds`가 완전히 동일한 연속 이벤트가 속도를 무한대로 만드는지 확인.